<a href="https://colab.research.google.com/github/prachimishraa/GenAI/blob/main/GenAI_Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 35.4 MB/s eta 0:00:00


In [ ]:
!pip install torchao==0.16.0

In [ ]:
!pip install -U "bitsandbytes>=0.46.1"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)


peft_model = get_peft_model(base_model, peft_config)
peft_model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [ ]:
from datasets import Dataset

custom_data = [
    {
        "question": "What is the policy for remote work at TechCorp?",
        "answer": "TechCorp allows up to 3 days of remote work per week with manager approval."
    },
    {
        "question": "How do I request time off?",
        "answer": "Time off must be requested via the HR Portal at least 14 days in advance."
    },
    {
        "question": "What is the refund window for SaaS products?",
        "answer": "Customers can claim a full refund within 30 days of subscription purchase."
    },
    {
        "question": "Who do I contact for IT support?",
        "answer": "Reach out to support@techcorp.com or open a ticket in Jira."
    }
]

dataset = Dataset.from_list(custom_data)

def format_prompts(example):
    messages = [
        {"role": "system", "content": "You are a helpful assistant for TechCorp."},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

formatted_dataset = dataset.map(format_prompts)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen_lora_results",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    num_train_epochs=10,
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=formatted_dataset,
    args=training_args,
)

trainer.train()

peft_model.save_pretrained("./fine_tuned_lora_model")
tokenizer.save_pretrained("./fine_tuned_lora_model")

Adding EOS to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
1,4.743123
2,4.743123
3,4.741458
4,4.295963
5,4.086774
6,3.890224
7,3.712157
8,3.546607
9,3.422410
10,3.347301


('./fine_tuned_lora_model/tokenizer_config.json',
 './fine_tuned_lora_model/chat_template.jinja',
 './fine_tuned_lora_model/tokenizer.json')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

def generate_response(model, tokenizer, query):
    messages = [
        {"role": "system", "content": "You are a helpful assistant for TechCorp."},
        {"role": "user", "content": query}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )

    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

raw_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_model = PeftModel.from_pretrained(raw_base_model, "./fine_tuned_lora_model")
merged_model = lora_model.merge_and_unload()

test_questions = [
    "What is the policy for remote work at TechCorp?",
    "How do I request time off?",
    "What is the refund window for SaaS products?",
    "Who do I contact for IT support?"
]

print("=" * 110)
print(f"{'QUESTION':<40} | {'ORIGINAL BASE MODEL':<32} | {'FINE-TUNED MODEL':<32}")
print("=" * 110)

for q in test_questions:
    orig_ans = generate_response(raw_base_model, tokenizer, q).replace('\n', ' ')
    ft_ans = generate_response(merged_model, tokenizer, q).replace('\n', ' ')

    orig_str = (orig_ans[:29] + "...") if len(orig_ans) > 32 else orig_ans
    ft_str = (ft_ans[:29] + "...") if len(ft_ans) > 32 else ft_ans

    print(f"{q:<40} | {orig_str:<32} | {ft_str:<32}")
print("=" * 110)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

QUESTION                                 | ORIGINAL BASE MODEL              | FINE-TUNED MODEL                
What is the policy for remote work at TechCorp? | As an AI language model, I do... | As an AI language model, I do...
How do I request time off?               | As an AI language model, I do... | As an AI language model, I do...
What is the refund window for SaaS products? | The refund window for SaaS (S... | The refund window for SaaS (S...
Who do I contact for IT support?         | If you need help with your IT... | If you need help with your IT...
